# A 2010–2020 timelapse of live performances logged on setlist.fm.

Every city is a live LED level meter that rises and falls with each year's performances, with a
white "peak hold" marker at its busiest year so far. A master meter along the bottom shows
the total across all cities. When 2020 arrives, the bars collapse but the peaks stay.

Inputs (made by concerts_download.ipynb, in the same folder):
  concert_cities.csv, concert_counts.csv
Natural Earth country outlines are downloaded automatically on the first run.

Outputs:
  concerts_timelapse.gif   for X/Bluesky/README
  concerts_timelapse.mp4   for LinkedIn/Instagram (needs: pip install imageio imageio-ffmpeg)

  pip install geopandas pandas numpy matplotlib pillow imageio imageio-ffmpeg

Run the cells from top to bottom. To tweak, edit the settings cell, re-run it, then re-run the last cell.

In [58]:
import os
import urllib.request

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

## Settings
Text, size, timing (`TWEEN`, `HOLD_MS`, `FINAL_HOLD_MS`), colours, meter sizes, insets, `LABEL_POS` for crowded labels, and `MIN_AVG_PER_YEAR` for cities with too little data.

In [60]:
CITIES_CSV = "concert_cities.csv"
COUNTS_CSV = "concert_counts.csv"
NE_FILE = "ne_50m_admin_0_countries.geojson"
NE_URL = ("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/"
          "geojson/ne_50m_admin_0_countries.geojson")
GIF_OUT = "concerts_timelapse.gif"
MP4_OUT = "concerts_timelapse.mp4"

TITLE = "Turn Down for What"
# {n} is filled in with the number of cities shown as meters (cities with too little
# data, shown as grey circles, aren't counted).
SUBTITLE = "Live performances logged in {n} cities, {y0}–{y1}"
CREDIT = "Data: setlist.fm API · Music: “On The Road To The Eighties” by Grand_Project / Roman Dudchyk, via Pixabay"
NOTE = ("Counts are artist sets logged by fans on setlist.fm, within each city's limits. "
        "Some of the rise reflects the site's growth, and search counts stop at 10,000 per city per year, "
        "so the biggest cities' peaks may be higher.")
# Size and timing
FIG_SIZE = (12, 7.5)      # inches; × DPI = pixels
DPI = 110                 # 12 × 110 = 1320 px wide
TWEEN = 8                 # in-between frames from one year to the next
FRAME_MS = 70             # length of each in-between frame
HOLD_MS = 650             # pause on each year
FINAL_HOLD_MS = 3500      # pause on the last year
MP4_FPS = 20
GIF_COLOURS = 96          # fewer colours = smaller GIF

# Dark "stage" theme
BG = "#0f1115"
LAND = "#22262d"
LAND_EDGE = "#2c313a"
INK = "#f2efe8"
INK_SOFT = "#a6a39b"
LED_OFF = "#1d2026"
PEAK = "#ffffff"
LED = ["#3ddc84"] * 6 + ["#c6e84a", "#f2d43d", "#f2a93d", "#f2713d", "#ef3b3b", "#ef3b3b"]
SEGMENTS = len(LED)

# City meters, in map metres (Equal Earth). Busier cities get taller meters.
BAR_W = 300_000
MAX_H = 2_300_000
MIN_FRAC = 0.4

# Zoom insets: (label, lon/lat box, position on main map xmin, ymin, xmax, ymax,
# label corner, meter size inside the inset)
INSETS = [
    ("North America", (-123, 28, -70, 52), (-17.2e6, 0.1e6, -11.4e6, 4.4e6), "bottom left", 0.55),
    ("Europe", (-17, 36, 31, 66), (-17.2e6, -7.9e6, -7.0e6, -0.5e6), "bottom left", 0.24),
    ("Asia", (95, -12, 145, 53), (3.6e6, -7.9e6, 10.6e6, -1.8e6), "bottom right", 0.45),
]
LABEL_SIZE = 7

# Cities averaging fewer logged performances a year than this (before the final year) are
# shown as grey "not enough data" dots instead of meters, and listed in the note.
MIN_AVG_PER_YEAR = 50
SHOW_SPARSE = False   # True = show those cities as grey circles; False = leave them out
SPARSE = "#5d626b"

# Where to put a city's name if it collides with a neighbour: "below" (default),
# "above", "left" or "right". Adjust after seeing your animation.
# A pair of numbers (km east, km north) moves the label out with a thin leader line,
# for crowded spots like the UK.
LABEL_POS = {
    "Toronto": "above", "New York": "below", "Chicago": "left", "Las Vegas": "above",
    "Los Angeles": "right",
    "Glasgow": (-480, 260), "Dublin": (-420, 40), "Manchester": (-560, -170),
    "London": (-600, -380), "Amsterdam": "above", "Brussels": "left",
    "Hamburg": "above", "Copenhagen": "right", "Berlin": "right", "Warsaw": "right",
    "Prague": "left", "Vienna": "right", "Oslo": "left", "Stockholm": "above",
    "Helsinki": "right",
    "Kuala Lumpur": "left", "Singapore": "right", "Bangkok": "left",
    "Ho Chi Minh City": "below", "Manila": "right", "Osaka": "below", "Tokyo": "right",
    "Seoul": "below", "Beijing": "left", "Taipei": "right", "Hong Kong": "left",
    "Tel Aviv": "left", "Cairo": "left", "Santiago": "left", "Buenos Aires": "right",
    "Melbourne": "left", "Sydney": "right",
}

## Data
Reads the CSVs from the download notebook.

In [62]:
def load():
    cities = pd.read_csv(CITIES_CSV, dtype={"city_id": str}).drop_duplicates("city_id")
    counts = pd.read_csv(COUNTS_CSV, dtype={"city_id": str})
    t = counts.pivot_table(index="city_id", columns="year", values="setlists",
                           aggfunc="sum").fillna(0)
    years = sorted(t.columns)
    t = t.join(cities.set_index("city_id")[["query", "country", "lat", "lon"]], how="inner")
    t = t.rename(columns={"query": "city"})
    pts = gpd.GeoSeries(gpd.points_from_xy(t["lon"], t["lat"]), crs="EPSG:4326").to_crs(8857)
    t["x"], t["y"] = pts.x.values, pts.y.values
    avg = t[years[:-1]].mean(axis=1)
    sparse = t[avg < MIN_AVG_PER_YEAR]
    if len(sparse):
        print("Too few logged performances to show as meters:", ", ".join(sparse["city"]))
    return t[avg >= MIN_AVG_PER_YEAR], sparse, years


def load_world():
    if not os.path.exists(NE_FILE):
        print("Downloading Natural Earth country outlines ...")
        urllib.request.urlretrieve(NE_URL, NE_FILE)
    w = gpd.read_file(NE_FILE)
    return w[w["ADM0_A3"] != "ATA"].to_crs(8857)


def box_xy(lonlat):
    # sample the whole outline: on a curved projection the corners alone aren't enough
    lon0, lat0, lon1, lat1 = lonlat
    lons = np.r_[np.linspace(lon0, lon1, 30), np.full(30, lon1),
                 np.linspace(lon1, lon0, 30), np.full(30, lon0)]
    lats = np.r_[np.full(30, lat0), np.linspace(lat0, lat1, 30),
                 np.full(30, lat1), np.linspace(lat1, lat0, 30)]
    b = gpd.GeoSeries.from_xy(lons, lats, crs="EPSG:4326").to_crs(8857)
    return b.x.min(), b.y.min(), b.x.max(), b.y.max()


def in_box(row, lonlat):
    lon0, lat0, lon1, lat1 = lonlat
    return lon0 <= row["lon"] <= lon1 and lat0 <= row["lat"] <= lat1

## Map and meters
Draws everything that doesn't move, plus a meter for each city.

In [64]:
class Meter:
    """One city's LED level meter: SEGMENTS rectangles plus a peak-hold marker."""

    def __init__(self, ax, x, y, height, scale, values, name):
        self.values = values.astype(float)
        self.peak_all = max(self.values.max(), 1)
        self.peak_seen = 0.0
        w = BAR_W * scale
        seg_h = height / SEGMENTS
        self.segs = []
        for s in range(SEGMENTS):
            r = Rectangle((x - w / 2, y + s * seg_h + seg_h * 0.14), w, seg_h * 0.72,
                          facecolor=LED_OFF, edgecolor="none", zorder=5)
            ax.add_patch(r)
            self.segs.append(r)
        self.peak = Rectangle((x - w / 2, y), w, seg_h * 0.22, facecolor=PEAK,
                              edgecolor="none", zorder=6, visible=False)
        ax.add_patch(self.peak)
        self.y, self.seg_h = y, seg_h
        pos = LABEL_POS.get(name, "below")
        if isinstance(pos, (tuple, list)):
            tx, ty = x + pos[0] * 1000, y + pos[1] * 1000
            ha, va = ("right" if pos[0] < 0 else "left"), "center"
            ax.plot([x, tx], [y, ty], color="#5d626b", linewidth=0.6, zorder=6)
        elif pos == "above":
            tx, ty, ha, va = x, y + height + seg_h * 0.5, "center", "bottom"
        elif pos == "left":
            tx, ty, ha, va = x - w * 0.9, y + seg_h * 1.5, "right", "center"
        elif pos == "right":
            tx, ty, ha, va = x + w * 0.9, y + seg_h * 1.5, "left", "center"
        else:
            tx, ty, ha, va = x, y - seg_h * 0.6, "center", "top"
        ax.text(tx, ty, name, ha=ha, va=va, fontsize=LABEL_SIZE, color=INK_SOFT, zorder=7,
                clip_on=True)

    def update(self, value):
        level = value / self.peak_all * SEGMENTS
        lit = int(round(level))
        if value > 0:
            lit = max(lit, 1)
        for s, r in enumerate(self.segs):
            r.set_facecolor(LED[s] if s < lit else LED_OFF)
        self.peak_seen = max(self.peak_seen, level)
        if self.peak_seen > 0:
            top = max(int(round(self.peak_seen)), 1)
            self.peak.set_y(self.y + top * self.seg_h - self.seg_h * 0.05)
            self.peak.set_visible(True)


def draw_sparse(ax, sparse, in_inset_lonlat=None):
    if not SHOW_SPARSE:
        return
    for _, r in sparse.iterrows():
        if in_inset_lonlat is not None and not in_box(r, in_inset_lonlat):
            continue
        ax.scatter([r["x"]], [r["y"]], s=26, facecolors="none", edgecolors=SPARSE,
                   linewidths=1.0, zorder=6)
        ax.text(r["x"], r["y"], f"  {r['city']}", ha="left", va="center",
                fontsize=LABEL_SIZE - 0.5, color=SPARSE, style="italic", zorder=7)


def build_scene(table, sparse, years, world):
    fig = plt.figure(figsize=FIG_SIZE, dpi=DPI, facecolor=BG)
    totals = table[years].sum(axis=1)
    frac = np.sqrt(totals / totals.max())
    heights = (MIN_FRAC + (1 - MIN_FRAC) * frac) * MAX_H

    fig.text(0.03, 0.955, TITLE, fontsize=24, fontweight="bold", color=INK, va="center")
    fig.text(0.03, 0.912, SUBTITLE.format(n=len(table), y0=years[0], y1=years[-1]), fontsize=12.5,
             color=INK_SOFT, va="center")

    ax = fig.add_axes([0.0, 0.17, 1.0, 0.70])
    world.plot(ax=ax, color=LAND, edgecolor=LAND_EDGE, linewidth=0.3)
    ax.set_xlim(-17.3e6, 17.3e6)
    ax.set_ylim(-8.0e6, 8.6e6)
    ax.set_aspect("equal")
    ax.set_axis_off()

    meters = []
    inside = table.apply(lambda r: any(in_box(r, i[1]) for i in INSETS), axis=1)
    for cid, r in table[~inside].sort_values("y", ascending=False).iterrows():
        meters.append(Meter(ax, r["x"], r["y"], heights[cid], 1.0, r[years].values, r["city"]))
    ax.scatter(table[inside]["x"], table[inside]["y"], s=4, color=LED[0], zorder=6)
    sparse_inside = sparse.apply(lambda r: any(in_box(r, i[1]) for i in INSETS), axis=1) \
        if len(sparse) else pd.Series(dtype=bool)
    draw_sparse(ax, sparse[~sparse_inside] if len(sparse) else sparse)

    for label, lonlat, (x0, y0, x1, y1), corner, scale in INSETS:
        bx0, by0, bx1, by1 = box_xy(lonlat)
        ins = ax.inset_axes([x0, y0, x1 - x0, y1 - y0], transform=ax.transData)
        world.plot(ax=ins, color=LAND, edgecolor=LAND_EDGE, linewidth=0.4)
        cx, cy = (bx0 + bx1) / 2, (by0 + by1) / 2
        aspect = (x1 - x0) / (y1 - y0)
        half_w = max((bx1 - bx0) / 2, (by1 - by0) / 2 * aspect)
        half_h = half_w / aspect
        ins.set_xlim(cx - half_w, cx + half_w)
        ins.set_ylim(cy - half_h, cy + half_h)
        ins.set_aspect("auto")
        ins.set_xticks([])
        ins.set_yticks([])
        ins.set_xlabel("")
        ins.set_ylabel("")
        ins.set_facecolor(BG)
        for sp in ins.spines.values():
            sp.set_edgecolor("#4a4f58")
            sp.set_linewidth(0.8)
        right, bottom = "right" in corner, "bottom" in corner
        ins.text(0.97 if right else 0.03, 0.04 if bottom else 0.96, label,
                 transform=ins.transAxes, fontsize=9, fontweight="bold", color=INK,
                 ha="right" if right else "left", va="bottom" if bottom else "top", zorder=10)
        ax.add_patch(Rectangle((bx0, by0), bx1 - bx0, by1 - by0, fill=False,
                               edgecolor="#4a4f58", linewidth=0.6, linestyle=(0, (3, 2)),
                               zorder=7))
        if len(sparse):
            draw_sparse(ins, sparse, in_inset_lonlat=lonlat)
        sub = table[table.apply(lambda r: in_box(r, lonlat), axis=1)]
        for cid, r in sub.sort_values("y", ascending=False).iterrows():
            meters.append(Meter(ins, r["x"], r["y"], heights[cid] * scale, scale,
                                r[years].values, r["city"]))

    # bottom panel: big year, master meter, timeline
    year_text = fig.text(0.03, 0.085, str(years[0]), fontsize=46, fontweight="bold",
                         color=INK, va="center")
    total_text = fig.text(0.20, 0.125, "", fontsize=11, color=INK_SOFT, va="center")
    master_ax = fig.add_axes([0.20, 0.06, 0.55, 0.045])
    master_ax.set_xlim(0, 1)
    master_ax.set_ylim(0, 1)
    master_ax.set_axis_off()
    n_master = 40
    master = []
    for i in range(n_master):
        colour_i = LED[min(int(i / n_master * SEGMENTS), SEGMENTS - 1)]
        r = Rectangle((i / n_master + 0.002, 0.1), 1 / n_master - 0.006, 0.8,
                      facecolor=LED_OFF, edgecolor="none")
        r.on_colour = colour_i
        master_ax.add_patch(r)
        master.append(r)
    master_peak = Rectangle((0, 0.0), 0.004, 1.0, facecolor=PEAK, edgecolor="none",
                            visible=False)
    master_ax.add_patch(master_peak)

    tl_ax = fig.add_axes([0.79, 0.055, 0.18, 0.09])
    tl_ax.set_xlim(years[0] - 0.5, years[-1] + 0.5)
    tl_ax.set_ylim(0, 1)
    tl_ax.set_axis_off()
    dots = tl_ax.scatter(years, [0.6] * len(years), s=22, color=LED_OFF, zorder=3)
    tl_ax.text(years[0], 0.05, str(years[0]), color=INK_SOFT, fontsize=8, ha="center")
    tl_ax.text(years[-1], 0.05, str(years[-1]), color=INK_SOFT, fontsize=8, ha="center")
    drop_text = fig.text(0.97, 0.18, "", fontsize=14, fontweight="bold", color="#ef3b3b",
                         ha="right", va="center")

    note = NOTE
    if len(sparse) and SHOW_SPARSE:
        note += (f" Grey circles: under {MIN_AVG_PER_YEAR} logged performances a year, too few "
                 "to show.")
    fig.text(0.03, 0.022, note, fontsize=7.5, color=INK_SOFT, va="center")
    fig.text(0.97, 0.955, CREDIT, fontsize=8, color=INK_SOFT, va="center", ha="right")

    yearly_total = table[years].sum().values.astype(float)
    state = dict(master=master, master_peak=master_peak, master_peak_seen=0.0,
                 total_max=yearly_total.max(), year_text=year_text, total_text=total_text,
                 dots=dots, drop_text=drop_text, yearly_total=yearly_total)
    return fig, meters, state

## Animation
Steps through the years and saves the GIF and MP4.

In [66]:
def ease(t):
    return t * t * (3 - 2 * t)  # smoothstep


def set_frame(meters, state, years, pos):
    """pos is a fractional index into years, e.g. 2.5 = halfway from years[2] to years[3]."""
    i0 = int(np.floor(pos))
    i1 = min(i0 + 1, len(years) - 1)
    f = ease(pos - i0)
    for m in meters:
        m.update(m.values[i0] * (1 - f) + m.values[i1] * f)
    total = state["yearly_total"][i0] * (1 - f) + state["yearly_total"][i1] * f
    level = total / state["total_max"]
    n = len(state["master"])
    lit = int(round(level * n))
    for k, r in enumerate(state["master"]):
        r.set_facecolor(r.on_colour if k < lit else LED_OFF)
    state["master_peak_seen"] = max(state["master_peak_seen"], level)
    state["master_peak"].set_x(min(state["master_peak_seen"], 0.996))
    state["master_peak"].set_visible(True)
    shown = i1 if f >= 0.5 else i0
    state["year_text"].set_text(str(years[shown]))
    state["total_text"].set_text(f"{total:,.0f} performances logged across {len(meters)} cities")
    colours = [LED[0] if k < shown else ("#ef3b3b" if k == shown else LED_OFF)
               for k in range(len(years))]
    state["dots"].set_color(colours)
    if shown == len(years) - 1 and len(years) > 1:
        prev, last = state["yearly_total"][-2], state["yearly_total"][-1]
        state["drop_text"].set_text(f"{years[-1]}: {(last / prev - 1) * 100:+.0f}% "
                                    f"vs {years[-2]}")
    else:
        state["drop_text"].set_text("")


def grab(fig):
    fig.canvas.draw()
    return Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3].copy())


def make_timelapse():
    table, sparse, years = load()
    world = load_world()
    fig, meters, state = build_scene(table, sparse, years, world)

    frames, durations = [], []
    for i in range(len(years)):
        set_frame(meters, state, years, i)
        frames.append(grab(fig))
        durations.append(FINAL_HOLD_MS if i == len(years) - 1 else HOLD_MS)
        if i < len(years) - 1:
            for k in range(1, TWEEN):
                set_frame(meters, state, years, i + k / TWEEN)
                frames.append(grab(fig))
                durations.append(FRAME_MS)
        print(f"  frames for {years[i]} done")
    plt.close(fig)

    pal = [f.quantize(colors=GIF_COLOURS, method=Image.Quantize.MEDIANCUT) for f in frames]
    pal[0].save(GIF_OUT, save_all=True, append_images=pal[1:], duration=durations, loop=0,
                optimize=True, disposal=1)
    print(f"Saved {GIF_OUT} ({os.path.getsize(GIF_OUT) / 1e6:.1f} MB, {len(frames)} frames)")

    try:
        import imageio.v2 as imageio
        import imageio_ffmpeg  # noqa: F401  (needed for MP4; pip install imageio-ffmpeg)
        with imageio.get_writer(MP4_OUT, format="FFMPEG", fps=MP4_FPS, codec="libx264",
                                quality=8, macro_block_size=8,
                                pixelformat="yuv420p") as w:
            for img, ms in zip(frames, durations):
                arr = np.asarray(img)
                h, wd = arr.shape[:2]
                arr = arr[:h - h % 2, :wd - wd % 2]
                for _ in range(max(1, round(ms / 1000 * MP4_FPS))):
                    w.append_data(arr)
        print(f"Saved {MP4_OUT}")
    except ImportError:
        print("GIF done. For an MP4 too: pip install imageio imageio-ffmpeg, then re-run.")
    except Exception as err:
        print(f"GIF done, but the MP4 failed ({err}). Try: pip install --upgrade imageio-ffmpeg")

## Run

In [68]:
make_timelapse()

Too few logged performances to show as meters: Mumbai, Ho Chi Minh City, Hong Kong, Lagos, Cairo
  frames for 2010 done
  frames for 2011 done
  frames for 2012 done
  frames for 2013 done
  frames for 2014 done
  frames for 2015 done
  frames for 2016 done
  frames for 2017 done
  frames for 2018 done
  frames for 2019 done
  frames for 2020 done
Saved concerts_timelapse.gif (1.5 MB, 81 frames)
Saved concerts_timelapse.mp4


In [72]:
import subprocess
import imageio_ffmpeg

MUSIC = "grand_project-on-the-road-to-the-eighties_loop1-177567.mp3"
MUSIC_OUT = "concerts_timelapse_music.mp4"
FADE_S = 2.0       # seconds the music takes to fade out
END_GAP_S = 1.0    # silence at the very end of the video

ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
n_frames, video_s = imageio_ffmpeg.count_frames_and_secs(MP4_OUT)
music_end = video_s - END_GAP_S                   # music finishes 1 s before the end
fade_start = max(0, music_end - FADE_S)

cmd = [ffmpeg, "-y", "-loglevel", "error",
       "-i", MP4_OUT,                              # the video
       "-stream_loop", "-1", "-i", MUSIC,          # the music, looped if it's too short
       "-filter_complex",
       f"[1:a]atrim=0:{music_end:.2f},afade=t=in:st=0:d=0.5,"
       f"afade=t=out:st={fade_start:.2f}:d={FADE_S},apad[a]",
       "-map", "0:v", "-map", "[a]",
       "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-shortest",
       MUSIC_OUT]
subprocess.run(cmd, check=True)
print(f"Saved {MUSIC_OUT}: music fades out at {fade_start:.1f}–{music_end:.1f}s, "
      f"then silence for the 2020 hold")

Saved concerts_timelapse_music.mp4: music fades out at 10.5–12.5s, then silence for the 2020 hold


In [74]:
# music downloaded from https://pixabay.com/music/synthwave-on-the-road-to-the-eighties-loop1-177567/
#Music: “On The Road To The Eighties” by Grand_Project / Roman Dudchyk, via Pixabay